In [ ]:
# [CELL 1] STUDENT CONFIGURATION 
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-1227"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
RANDOM_SEED = int(id_digits) % (2**32)

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Random Seed:", RANDOM_SEED)

Student: Solis TUPM-26-1227
Seed Digit: 7
ID Sum: 20
Random Seed: 261227


In [2]:
# [CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

for path in [measurement_path, label_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("Dataset files verified.")

Workspace: C:\Users\n0n3\Lab7_NumPy_Pandas_Solis\Lab7_NumPy_Pandas_Additional_Exercises
Dataset files verified.


In [3]:
# [CELL 3] LOAD DATA AND IDENTIFY USABLE FEATURES
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw_measurements.shape[0] == len(labels)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw_measurements.shape
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0

observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)

usable_mask = ~(all_missing | constant)
usable_indices = np.flatnonzero(usable_mask)

pass_mask = labels == -1
fail_mask = labels == 1

print("Shape:", raw_measurements.shape)
print("Pass:", pass_mask.sum(), "| Fail:", fail_mask.sum())
print("Usable Features:", len(usable_indices))

Shape: (1567, 590)
Pass: 1463 | Fail: 104
Usable Features: 474


In [4]:
# [CELL 4] LABELED DATAFRAME WITH ORIGINAL RECORD IDS
meta = pd.DataFrame({
    "Example_ID": np.arange(n_records),          # original SECOM row identifier
    "Result": labels,
    "Outcome": np.where(labels == -1, "Pass", "Fail"),
})
df = pd.concat(
    [meta, pd.DataFrame(raw_measurements, columns=feature_names)], axis=1
)

print("DataFrame Shape:", df.shape)
print("Pass Records:", pass_mask.sum(), "| Fail Records:", fail_mask.sum())

DataFrame Shape: (1567, 593)
Pass Records: 1463 | Fail Records: 104


In [5]:
# [CELL 5] REPRODUCIBLE BALANCED SAMPLE
rng = np.random.default_rng(RANDOM_SEED)

# Same five-feature draw used in the main activity (Cell 11)
feature_positions = rng.choice(len(usable_indices), size=5, replace=False)
selected_indices = usable_indices[feature_positions]
selected_names = feature_names[selected_indices].tolist()

group_size = min(20 + ID_SUM, int(pass_mask.sum()), int(fail_mask.sum()))

pass_rows = rng.choice(np.flatnonzero(pass_mask), size=group_size, replace=False)
fail_rows = rng.choice(np.flatnonzero(fail_mask), size=group_size, replace=False)
sample_rows = np.sort(np.concatenate([pass_rows, fail_rows]))

print("Selected Features:", selected_names)
print("Records per Outcome:", group_size)
print("Balanced Sample Size:", len(sample_rows))

Selected Features: ['Feature_050', 'Feature_020', 'Feature_019', 'Feature_204', 'Feature_076']
Records per Outcome: 40
Balanced Sample Size: 80


In [6]:
# [CELL 6] MEDIAN IMPUTATION (medians from the FULL dataset)
full_medians = pd.Series(
    np.nanmedian(raw_measurements[:, selected_indices], axis=0),
    index=selected_names
)

prepared = df[selected_names].fillna(full_medians)

sample = pd.concat(
    [df[["Example_ID", "Result", "Outcome"]], prepared], axis=1
).loc[sample_rows].copy()

print("Missing in Sample Before:", int(df.loc[sample_rows, selected_names].isna().sum().sum()))
print("Missing in Sample After:", int(sample[selected_names].isna().sum().sum()))
print("Full-Dataset Medians:", full_medians.to_numpy())

Missing in Sample Before: 0
Missing in Sample After: 0
Full-Dataset Medians: [631.3709   1.406   12.4996   0.1582  -0.0289]


In [7]:
# [CELL 7] GROUPED STATISTICS WITH groupby() AND agg()
grouped = sample.groupby("Outcome")[selected_names].agg(
    ["count", "mean", "median", "std"]          # std defaults to ddof=1
)

counts  = grouped.xs("count",  axis=1, level=1)
means   = grouped.xs("mean",   axis=1, level=1)
medians = grouped.xs("median", axis=1, level=1)
sds     = grouped.xs("std",    axis=1, level=1)

print(grouped.round(4).T.to_string())

Outcome                  Fail       Pass
Feature_050 count     40.0000    40.0000
            mean     631.4448   633.0475
            median   629.8400   632.3336
            std        7.7914     9.1562
Feature_020 count     40.0000    40.0000
            mean       1.4066     1.4034
            median     1.4068     1.4060
            std        0.0131     0.0151
Feature_019 count     40.0000    40.0000
            mean      12.5151    12.4119
            median    12.5014    12.4864
            std        0.0575     0.4249
Feature_204 count     40.0000    40.0000
            mean     250.1538   250.1638
            median     0.1680     0.1708
            std     1580.7449  1580.7148
Feature_076 count     40.0000    40.0000
            mean      -0.0391    -0.0333
            median    -0.0410    -0.0330
            std        0.0399     0.0300


In [8]:
# [CELL 8] ABSOLUTE AND STANDARDIZED MEAN DIFFERENCE
n_f, n_p = counts.loc["Fail"].to_numpy(), counts.loc["Pass"].to_numpy()
s_f, s_p = sds.loc["Fail"].to_numpy(),    sds.loc["Pass"].to_numpy()

abs_diff = np.abs(means.loc["Fail"].to_numpy() - means.loc["Pass"].to_numpy())

pooled_sd = np.sqrt(((n_f - 1) * s_f**2 + (n_p - 1) * s_p**2) / (n_f + n_p - 2))

# Zero pooled SD -> standardized difference set to 0 (no division by zero)
std_diff = np.divide(abs_diff, pooled_sd,
                     out=np.zeros_like(abs_diff), where=pooled_sd > 0)

print("Absolute Mean Difference:", abs_diff)
print("Pooled SD:", pooled_sd)
print("Standardized Difference:", std_diff)
print("Zero Pooled SD Features:", int((pooled_sd == 0).sum()))

Absolute Mean Difference: [1.6027 0.0032 0.1032 0.01   0.0059]
Pooled SD: [   8.5012    0.0142    0.3032 1580.7299    0.0353]
Standardized Difference: [0.1885 0.2233 0.3404 0.     0.1658]
Zero Pooled SD Features: 0


In [9]:
# [CELL 9] RANK FEATURES AND EXPORT
feature_comparison = pd.DataFrame({
    "Feature": selected_names,
    "Pass_Count": n_p, "Fail_Count": n_f,
    "Pass_Mean": means.loc["Pass"].to_numpy(), "Fail_Mean": means.loc["Fail"].to_numpy(),
    "Pass_Median": medians.loc["Pass"].to_numpy(), "Fail_Median": medians.loc["Fail"].to_numpy(),
    "Pass_SD": s_p, "Fail_SD": s_f,
    "Abs_Mean_Diff": abs_diff, "Pooled_SD": pooled_sd,
    "Abs_Std_Mean_Diff": std_diff,
}).sort_values("Abs_Std_Mean_Diff", ascending=False).reset_index(drop=True)
feature_comparison.insert(0, "Rank", np.arange(1, len(feature_comparison) + 1))

sample_path = OUTPUT_DIR / "balanced_sample.csv"
comparison_path = OUTPUT_DIR / "feature_comparison.csv"
sample.to_csv(sample_path, index=False)
feature_comparison.to_csv(comparison_path, index=False)

print(feature_comparison[["Rank", "Feature", "Abs_Mean_Diff", "Abs_Std_Mean_Diff"]]
      .round(4).to_string(index=False))

 Rank     Feature  Abs_Mean_Diff  Abs_Std_Mean_Diff
    1 Feature_019         0.1032             0.3404
    2 Feature_020         0.0032             0.2233
    3 Feature_050         1.6027             0.1885
    4 Feature_076         0.0059             0.1658
    5 Feature_204         0.0100             0.0000


In [10]:
# [CELL 10] VERIFY NUMPY-PANDAS STATISTICS AND EXPORTS
X = sample[selected_names].to_numpy()
is_fail = (sample["Outcome"] == "Fail").to_numpy()

np_ok = True
for group, name in [(is_fail, "Fail"), (~is_fail, "Pass")]:
    block = X[group]
    np_ok &= np.array_equal(block.shape[0] * np.ones(X.shape[1]), counts.loc[name].to_numpy())
    np_ok &= np.allclose(block.mean(axis=0),         means.loc[name].to_numpy())
    np_ok &= np.allclose(np.median(block, axis=0),   medians.loc[name].to_numpy())
    np_ok &= np.allclose(block.std(axis=0, ddof=1),  sds.loc[name].to_numpy())

loaded_sample = pd.read_csv(sample_path)
loaded_cmp = pd.read_csv(comparison_path)

ids_ok = np.array_equal(
    loaded_sample["Result"].to_numpy(), labels[loaded_sample["Example_ID"].to_numpy()]
) and np.allclose(
    loaded_sample[selected_names].to_numpy(),
    df.loc[loaded_sample["Example_ID"], selected_names].fillna(full_medians).to_numpy()
)
csv_ok = (len(loaded_sample) == 2 * group_size
        and np.allclose(loaded_cmp["Abs_Std_Mean_Diff"], feature_comparison["Abs_Std_Mean_Diff"]))
balanced_ok = (sample["Outcome"].value_counts() == group_size).all()

stats_match = bool(np_ok and ids_ok and csv_ok and balanced_ok)

print("NumPy Statistics Match:", bool(np_ok))
print("Record IDs / Labels Verified:", bool(ids_ok))
print("CSV Reload Verified:", bool(csv_ok))
print("Balanced Groups:", bool(balanced_ok))

NumPy Statistics Match: True
Record IDs / Labels Verified: True
CSV Reload Verified: True
Balanced Groups: True


In [11]:
# [CELL 11] ASSESSMENT DATA (one line per answer)
top = feature_comparison.iloc[0]

print("Balanced Sample Size:", f"{len(sample)} ({group_size}+{group_size})")
print("Selected Features:", ", ".join(str(i) for i in np.sort(selected_indices)))
print("Highest-Ranked Feature:", int(top["Feature"].split("_")[1]))
print("Absolute Standardized Mean Difference:", f"{top['Abs_Std_Mean_Diff']:.4f}")
print("NumPy-Pandas Statistics Match:", "Yes" if stats_match else "No")

Balanced Sample Size: 80 (40+40)
Selected Features: 19, 20, 50, 76, 204
Highest-Ranked Feature: 19
Absolute Standardized Mean Difference: 0.3404
NumPy-Pandas Statistics Match: Yes
